# Models

Three analyses, one pipeline each:

1. **Union density** (blue charts)
2. **Electoral democracy** (orange charts)
3. **Electoral democracy without V-Dem predictors** (green charts)

Each section fits an OLS on the training countries for its coefficients, then scores OLS and a random forest with **nested**, grouped cross-validation: the forest is re-tuned inside every outer fold, using only that fold's training countries, so no country it is scored on ever influenced its settings. Permutation importances come from the same folds.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
# OLS is imported directly: statsmodels.api also loads statsmodels.tsa.statespace,
# whose compiled _conventional DLL is blocked by Windows Application Control
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# NOTE: tuning.ipynb has an identical copy of this block
df = pd.read_csv("../data/merged.csv")

ID_COLS = ["iso3", "country", "Unnamed: 0"]
DROP = ["emp_services_pct",     # emp shares sum to exactly 100
        "services_va_pct_gdp",  # value-added shares nearly sum to 100
        "emp_pop_ratio",        # ≈ lfp_rate × (1 − unemployment)
        "self_employed_pct",    # r = 0.996 with vulnerable_emp_pct
        "exports_pct_gdp",      # r = 0.987 with trade_pct_gdp
        "liberal_democracy"]    # r = 0.981 with electoral_democracy
# V-Dem builds electoral_democracy (v2x_polyarchy) from freedom_of_association
# (v2x_frassoc_thick), which is itself built from cso_entry_exit and
# cso_repression; civil_society_index shares those two inputs (codebook v16).
# Predicting democracy from them would be predicting the index from its parts
DEM_INGREDIENTS = ["freedom_of_association", "cso_entry_exit",
                   "cso_repression", "civil_society_index"]
# every V-Dem column in merged.csv (VDEM_NAMES in cleaning.ipynb), for the
# democracy model with no V-Dem predictors at all
VDEM_COLS = ["freedom_of_association", "civil_society_index", "electoral_democracy",
             "liberal_democracy", "cso_repression", "cso_entry_exit",
             "egalitarian_component", "equal_resource_distribution",
             "power_by_socioeconomic_position"]

cols_union = [c for c in df if c not in ID_COLS + ["union_density"] + DROP]
cols_dem = [c for c in df if c not in ID_COLS + ["electoral_democracy"] + DROP + DEM_INGREDIENTS]
cols_dem_novdem = [c for c in df if c not in ID_COLS + VDEM_COLS + DROP]

df = df.dropna().reset_index(drop=True)

random_state = 42

# one train/test split: models.ipynb fits its OLS summaries on the training
# rows, and tuning.ipynb tunes on the training countries only. whole countries
# are held out, so no country appears in both halves
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
train_idx, test_idx = next(splitter.split(df, groups=df["iso3"]))
train, test = df.loc[train_idx], df.loc[test_idx]

print(f"train: {len(train):4d} rows, {train['iso3'].nunique():3d} countries")
print(f"test:  {len(test):4d} rows, {test['iso3'].nunique():3d} countries")
print("countries in both:", len(set(train["iso3"]) & set(test["iso3"])))
print(f"\nfeatures -- union density: {len(cols_union)} | democracy: {len(cols_dem)} | "
      f"democracy, no V-Dem: {len(cols_dem_novdem)}")

In [ ]:
# shared by every section, so each analysis gets exactly the same models,
# scoring, and charts
BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#008300"   # sections 1, 2, 3
MUTED, SURFACE = "#52514e", "#fcfcfb"
CV_REPEATS, CV_FOLDS = 3, 5

# the random forest's settings are searched over this grid (the same one as
# tuning.ipynb) inside every outer CV fold
GRID = {
    "max_features": [0.2, 0.33, 0.5, 1.0],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_samples": [None, 0.7],
}
SEARCH_TREES, FINAL_TREES = 300, 1000


def scale(cols):
    """Scaled training features for the OLS summaries."""
    scaler = StandardScaler().fit(train[cols])
    return pd.DataFrame(scaler.transform(train[cols]), columns=cols, index=train.index)


def tuned_forest(X, y, groups):
    """Grid-search the forest on these rows only, with inner folds grouped by
    country, then refit the winning settings with more trees."""
    search = GridSearchCV(
        RandomForestRegressor(n_estimators=SEARCH_TREES, random_state=random_state, n_jobs=1),
        GRID,
        cv=GroupKFold(CV_FOLDS),
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    ).fit(X, y, groups=groups)
    forest = RandomForestRegressor(n_estimators=FINAL_TREES, random_state=random_state,
                                   n_jobs=-1, **search.best_params_)
    return forest.fit(X, y), search.best_params_


def nested_cv(X, y, groups):
    """Scores and permutation importances from nested, grouped cross-validation.

    Outer loop: 5 folds of whole countries, repeated with the countries
    reshuffled, so every country is tested once per repeat. A single
    train/test split would rest on 20 test countries, which is far too few --
    the same model scored anywhere from -2.06 to 0.50 depending on the draw.

    Inner loop: the forest is tuned on the outer fold's training countries
    only, so the countries it is scored on never influenced its settings
    (Neunhoeffer & Sternberg 2019). OLS and the baseline have nothing to tune.
    """
    names = ["baseline", "OLS", "random forest"]
    rows, importances, chosen = [], [], []
    for seed in range(CV_REPEATS):
        oof = {name: np.zeros(len(y)) for name in names}
        # same folds for every model, so the comparison is paired
        cv = GroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)
        for tr, te in cv.split(X, y, groups):
            X_tr, y_tr, X_te, y_te = X.iloc[tr], y.iloc[tr], X.iloc[te], y.iloc[te]
            forest, best = tuned_forest(X_tr, y_tr, groups.iloc[tr])
            chosen.append(str(best))

            fitted = {
                "baseline": DummyRegressor(strategy="mean").fit(X_tr, y_tr),
                "OLS": make_pipeline(StandardScaler(), LinearRegression()).fit(X_tr, y_tr),
                "random forest": forest,
            }
            for name, model in fitted.items():
                oof[name][te] = model.predict(X_te)

            # importance measured on countries the tuned forest never saw
            r = permutation_importance(forest, X_te, y_te, n_repeats=10,
                                       scoring="neg_root_mean_squared_error",
                                       random_state=0, n_jobs=-1)
            importances.append(pd.Series(r.importances_mean, index=X.columns))

        # pool the out-of-fold predictions and score once per repeat --
        # averaging each fold's R2 instead is dragged down by folds with
        # little spread to explain
        for name in names:
            rows.append({
                "model": name,
                "seed": seed,
                "RMSE": root_mean_squared_error(y, oof[name]),
                "MAE": mean_absolute_error(y, oof[name]),
                "R2": r2_score(y, oof[name]),
            })

    scores = pd.DataFrame(rows)
    table = (scores.groupby("model")[["RMSE", "MAE", "R2"]]
             .agg(["mean", "std"]).round(3)
             .reindex(names))
    print(f"nested {CV_FOLDS}-fold grouped CV, {CV_REPEATS} repeats, "
          f"{len(X)} rows / {groups.nunique()} countries\n")
    print(table.to_string())

    wide = scores.pivot(index="seed", columns="model", values="R2")
    print(f"\nforest beat OLS in {(wide['random forest'] > wide['OLS']).sum()} of {CV_REPEATS} repeats")

    picks = pd.Series(chosen).value_counts()
    print(f"\nforest settings chosen in the {len(chosen)} outer folds "
          f"({len(picks)} different combinations); most common:")
    print(picks.head(3).to_string())
    return scores, pd.DataFrame(importances)


def importance_table(imp):
    return (pd.DataFrame({"mean": imp.mean(), "sd": imp.std(), "folds_positive": (imp > 0).sum()})
            .sort_values("mean", ascending=False))


def importance_plot(imp, color, title, unit, top_n=10):
    """One dot per CV fold, a diamond for the mean, and the number of folds
    where shuffling the feature made predictions worse."""
    stats = importance_table(imp)
    top = stats.head(top_n).index[::-1]   # largest ends up at the top

    fig = go.Figure()
    for feat in top:
        fig.add_scatter(x=imp[feat], y=[feat] * len(imp), mode="markers", showlegend=False,
                        marker=dict(size=9, color=color, opacity=0.35,
                                    line=dict(width=1, color=SURFACE)),
                        hovertemplate="%{y}<br>fold: %{x:.3f}<extra></extra>")

    fig.add_scatter(x=stats.loc[top, "mean"], y=list(top), mode="markers", showlegend=False,
                    marker=dict(size=14, color=color, symbol="diamond",
                                line=dict(width=1.5, color=SURFACE)),
                    hovertemplate="%{y}<br><b>mean: %{x:.3f}</b><extra></extra>")

    fig.add_vline(x=0, line_width=1.5, line_color=MUTED)
    for feat in top:
        fig.add_annotation(x=1.02, xref="paper", y=feat, showarrow=False,
                           text=f"{stats.loc[feat, 'folds_positive']}/{len(imp)}",
                           font=dict(size=11, color=MUTED))

    fig.update_layout(
        title=f"{title}: permutation importance across {len(imp)} grouped CV folds"
              f"<br><sub>{unit} when the feature is shuffled · diamond = mean · "
              "right column = folds where shuffling hurt</sub>",
        xaxis=dict(title=unit, gridcolor="#eceae5", zeroline=False),
        yaxis=dict(gridcolor="rgba(0,0,0,0)"),
        plot_bgcolor=SURFACE, paper_bgcolor=SURFACE,
        margin=dict(l=210, r=70, t=80, b=50), height=430,
    )
    return fig

## 1. Predicting union density

In [ ]:
X_train_u = scale(cols_union)
y_train_u = train["union_density"]

In [ ]:
ols_union = OLS(y_train_u, add_constant(X_train_u)).fit(
    cov_type="cluster", cov_kwds={"groups": train["iso3"]}
)
print(ols_union.summary())

In [ ]:
cv_union, imp_union = nested_cv(df[cols_union], df["union_density"], df["iso3"])

In [ ]:
importance_table(imp_union).round(3)

In [ ]:
importance_plot(imp_union, BLUE, "Union density", "RMSE points lost")

## 2. Predicting electoral democracy

Same models and charts as section 1, with V-Dem's `electoral_democracy` (0–1) as the target and `union_density` as one of the predictors. The four `DEM_INGREDIENTS` features are left out because V-Dem builds `electoral_democracy` from them.

In [ ]:
X_train_d = scale(cols_dem)
y_train_d = train["electoral_democracy"]

In [ ]:
ols_dem = OLS(y_train_d, add_constant(X_train_d)).fit(
    cov_type="cluster", cov_kwds={"groups": train["iso3"]}
)
print(ols_dem.summary())

In [ ]:
cv_dem, imp_dem = nested_cv(df[cols_dem], df["electoral_democracy"], df["iso3"])

In [ ]:
importance_table(imp_dem).round(3)

In [ ]:
importance_plot(imp_dem, ORANGE, "Electoral democracy", "RMSE lost (index, 0–1)")

## 3. Predicting electoral democracy without V-Dem predictors

Section 2 again, but with every V-Dem column (`VDEM_COLS`) removed from the predictors — including the three equality indices, which carried most of section 2's importance and are themselves V-Dem democracy indices rated by the same experts. What's left is World Bank variables plus `union_density`, so this is the cleanest test of whether economic structure and unions predict democracy.

In [ ]:
X_train_nv = scale(cols_dem_novdem)
y_train_nv = train["electoral_democracy"]

In [ ]:
ols_novdem = OLS(y_train_nv, add_constant(X_train_nv)).fit(
    cov_type="cluster", cov_kwds={"groups": train["iso3"]}
)
print(ols_novdem.summary())

In [ ]:
cv_novdem, imp_novdem = nested_cv(df[cols_dem_novdem], df["electoral_democracy"], df["iso3"])

In [ ]:
importance_table(imp_novdem).round(3)

In [ ]:
importance_plot(imp_novdem, GREEN, "Electoral democracy, no V-Dem", "RMSE lost (index, 0–1)")